# Experiments — Electricity Load Forecasting


## Imports & Setup

In [1]:
import sys
sys.path.append("..") 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

from src.experiments import config, data, features, cv, hpo
from src.experiments.mlflow_utils import log_cv_run
from src.inference import feature_engineering as fe

config.init_mlflow()


/teamspace/studios/this_studio/electricity-distribution-forecast/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Data & Split

In [18]:
df = data.load_raw_data("../data/interim/df_core_features.parquet")
df = features.add_base_features(df)

df = features.drop_base_feature_warmup(df)

train, test, TRAIN_END, TEST_START = data.chronological_split(df, purge_days=config.PURGE_DAYS)
train, test = features.attach_holiday_names(train, test)



print(f"Train: {len(train):,} rows | {train['timestamp'].min()} -> {train['timestamp'].max()}")
print(f"Test:  {len(test):,} rows | {test['timestamp'].min()} -> {test['timestamp'].max()}")


Dropped 24 warm-up row(s) for ['temp_c_roll_std_72', 'temp_change_vs_lag24']
Train: 83,931 rows | 2016-01-09 00:00:00+00:00 -> 2025-08-06 05:00:00+00:00
Test:  8,587 rows | 2025-08-13 05:00:00+00:00 -> 2026-08-05 23:00:00+00:00


## Baseline: Constant & Persistence

In [19]:
baseline_oof = cv.run_naive_baselines(train, config.FOLD_BOUNDARIES, config.TARGET)
for name, oof in baseline_oof.items():
    m = cv.compute_oof_metrics(train, oof, config.TARGET)
    print(f"{name} OOF RMSE: {m['rmse']:.2f}")


Fold 1  |  Constant RMSE: 1806.95  |  Persistence RMSE: 772.98
Fold 2  |  Constant RMSE: 2331.61  |  Persistence RMSE: 811.14
Fold 3  |  Constant RMSE: 2389.79  |  Persistence RMSE: 804.75
Fold 4  |  Constant RMSE: 2585.27  |  Persistence RMSE: 800.56
Fold 5  |  Constant RMSE: 2592.49  |  Persistence RMSE: 843.63
constant OOF RMSE: 2358.82
persistence OOF RMSE: 806.93


## Linear Regression (raw features)

In [20]:
result_lr = cv.run_raw_cv(
    train, config.FOLD_BOUNDARIES, config.TARGET, config.FEATURES,
    model_builder=LinearRegression, scale_X=True,
)
metrics_lr = cv.compute_oof_metrics(train, result_lr["oof"], config.TARGET)
print(f"Linear Model, CV OOF RMSE: {metrics_lr['rmse']:.2f}  MAPE: {metrics_lr['mape']:.2f}%")

log_cv_run(
    "Baseline_Linear_Regression",
    params={"model_type": "LinearRegression", "features": "raw_features", "cv_strategy": "5-fold-expanding"},
    metrics={"cv_oof_rmse": metrics_lr["rmse"], "cv_oof_mape": metrics_lr["mape"]},
)


Fold 1 RMSE: 740.44
Fold 2 RMSE: 766.35
Fold 3 RMSE: 747.30
Fold 4 RMSE: 761.37
Fold 5 RMSE: 795.27
Linear Model, CV OOF RMSE: 762.38  MAPE: 6.47%


Logged 'Baseline_Linear_Regression' — cv_oof_rmse: 762.38, cv_oof_mape: 6.47
🏃 View run Baseline_Linear_Regression at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2/runs/e0c2410c3526479298fe0dcfbe9bad71
🧪 View experiment at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2


## XGBoost (Baseline, raw features)

In [21]:
xgb_baseline_params = dict(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1)

result_xgb_base = cv.run_raw_cv(
    train, config.FOLD_BOUNDARIES, config.TARGET, config.FEATURES,
    model_builder=lambda: xgb.XGBRegressor(**xgb_baseline_params), scale_X=True,
)
metrics_xgb_base = cv.compute_oof_metrics(train, result_xgb_base["oof"], config.TARGET)
print(f"XGBoost Baseline, CV OOF RMSE: {metrics_xgb_base['rmse']:.2f}  MAPE: {metrics_xgb_base['mape']:.2f}%")

log_cv_run(
    "Baseline_XGBoost_Raw",
    params={"model_type": "XGBoost", "features": "raw_features", "cv_strategy": "5-fold-expanding", **xgb_baseline_params},
    metrics={"cv_oof_rmse": metrics_xgb_base["rmse"], "cv_oof_mape": metrics_xgb_base["mape"]},
)


Fold 1 RMSE: 457.04
Fold 2 RMSE: 618.83
Fold 3 RMSE: 760.06
Fold 4 RMSE: 690.77
Fold 5 RMSE: 683.67
XGBoost Baseline, CV OOF RMSE: 650.27  MAPE: 6.33%
Logged 'Baseline_XGBoost_Raw' — cv_oof_rmse: 650.27, cv_oof_mape: 6.33
🏃 View run Baseline_XGBoost_Raw at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2/runs/a9f88ed677df4b8f85351168ac7c2a19
🧪 View experiment at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2


## Trend + XGBoost-on-Residual (v1)
Raw features, no derived flags — `fold_feature_fn=None`.

In [22]:
default_gbm_params = dict(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1)

result_v1 = cv.run_trend_plus_cv(
    train, config.FOLD_BOUNDARIES, config.TARGET, config.FEATURES,
    model_builder=lambda: xgb.XGBRegressor(**default_gbm_params),
    fold_feature_fn=None,
)
metrics_v1 = cv.compute_oof_metrics(train, result_v1["oof"], config.TARGET)
print(f"Trend + XGB (v1), CV OOF RMSE: {metrics_v1['rmse']:.2f}  MAPE: {metrics_v1['mape']:.2f}%")

log_cv_run(
    "Trend_XGB_v1",
    params={"model_type": "Trend+XGBoost", "experiment_version": "v1", "features": "raw_features + trend_idx",
            "cv_strategy": "5-fold-expanding", **default_gbm_params},
    metrics={"cv_oof_rmse": metrics_v1["rmse"], "cv_oof_mape": metrics_v1["mape"]},
)


Fold 1 RMSE: 410.30
Fold 2 RMSE: 419.62
Fold 3 RMSE: 436.16
Fold 4 RMSE: 466.92
Fold 5 RMSE: 412.67
Trend + XGB (v1), CV OOF RMSE: 429.67  MAPE: 3.67%
Logged 'Trend_XGB_v1' — cv_oof_rmse: 429.67, cv_oof_mape: 3.67
🏃 View run Trend_XGB_v1 at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2/runs/86ac3a941cff44adb456584fe17f9f00
🧪 View experiment at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2


## Trend + XGBoost + Extreme-Event Features (v2)
Same extreme-event fold features as v3 below — only the *feature set passed to the model* differs (`FEATURES_V2` excludes `temp_change_vs_lag24` / `is_high_precip_event`).

In [23]:
result_v2 = cv.run_trend_plus_cv(
    train, config.FOLD_BOUNDARIES, config.TARGET, config.FEATURES_V2,
    model_builder=lambda: xgb.XGBRegressor(**default_gbm_params),
    fold_feature_fn=features.extreme_event_fold_features,
)
metrics_v2 = cv.compute_oof_metrics(train, result_v2["oof"], config.TARGET)
print(f"Trend + XGB + extreme events (v2), CV OOF RMSE: {metrics_v2['rmse']:.2f}  (v1: {metrics_v1['rmse']:.2f})")

log_cv_run(
    "Trend_XGB_v2",
    params={"model_type": "Trend+XGBoost", "experiment_version": "v2",
            "features": "FEATURES_V2 (extreme events + holiday interaction)",
            "cv_strategy": "5-fold-expanding", "extreme_heat_quantile": 0.95, "extreme_cold_quantile": 0.05,
            **default_gbm_params},
    metrics={"cv_oof_rmse": metrics_v2["rmse"], "cv_oof_mape": metrics_v2["mape"]},
)


Fold 1 RMSE: 411.39


Fold 2 RMSE: 421.60
Fold 3 RMSE: 417.61
Fold 4 RMSE: 467.94
Fold 5 RMSE: 403.08
Trend + XGB + extreme events (v2), CV OOF RMSE: 424.96  (v1: 429.67)
Logged 'Trend_XGB_v2' — cv_oof_rmse: 424.96, cv_oof_mape: 3.65
🏃 View run Trend_XGB_v2 at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2/runs/e3dc8d7293714d5d8d4681959658073b
🧪 View experiment at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2


## Model Sweep on v3 Features (XGBoost / LightGBM / CatBoost / KNN)
This replaces four separate copy-pasted cells — same CV runner, only the model builder changes.

In [24]:
v3_fold_features = features.extreme_event_fold_features

model_configs = {
    "xgb_v3":      dict(builder=lambda: xgb.XGBRegressor(**default_gbm_params), scale_X=False,
                         mlflow_name="Trend_XGB_v3", model_label="Trend+XGBoost",
                         hp=default_gbm_params),
    "lgbm_v3":     dict(builder=lambda: lgb.LGBMRegressor(**default_gbm_params, verbosity=-1), scale_X=False,
                         mlflow_name="Trend_LightGBM_v3", model_label="Trend+LightGBM",
                         hp=default_gbm_params),
    "catboost_v3": dict(builder=lambda: cb.CatBoostRegressor(
                             iterations=500, learning_rate=0.05, depth=6,
                             random_state=42, thread_count=-1, verbose=0), scale_X=False,
                         mlflow_name="Trend_CatBoost_v3", model_label="Trend+CatBoost",
                         hp=dict(iterations=500, learning_rate=0.05, depth=6)),
    "knn_v3":      dict(builder=lambda: KNeighborsRegressor(n_neighbors=15, weights="distance", n_jobs=-1), scale_X=True,
                         mlflow_name="Trend_KNN_v3", model_label="Trend+KNN",
                         hp=dict(n_neighbors=15, weights="distance")),
}

v3_results, v3_metrics = {}, {}
for name, cfg in model_configs.items():
    print(f"--- {name} ---")
    result = cv.run_trend_plus_cv(
        train, config.FOLD_BOUNDARIES, config.TARGET, config.FEATURES_V3,
        model_builder=cfg["builder"], fold_feature_fn=v3_fold_features, scale_X=cfg["scale_X"],
    )
    m = cv.compute_oof_metrics(train, result["oof"], config.TARGET)
    v3_results[name] = result
    v3_metrics[name] = m
    print(f"{cfg['model_label']} (v3), CV OOF RMSE: {m['rmse']:.2f}  MAPE: {m['mape']:.2f}%\n")

    log_cv_run(
        cfg["mlflow_name"],
        params={"model_type": cfg["model_label"], "experiment_version": "v3",
                "features": "FEATURES_V3", "cv_strategy": "5-fold-expanding", **cfg["hp"]},
        metrics={"cv_oof_rmse": m["rmse"], "cv_oof_mape": m["mape"]},
    )


--- xgb_v3 ---


Fold 1 RMSE: 407.80
Fold 2 RMSE: 380.97
Fold 3 RMSE: 386.74
Fold 4 RMSE: 446.89
Fold 5 RMSE: 379.67
Trend+XGBoost (v3), CV OOF RMSE: 401.24  MAPE: 3.48%

Logged 'Trend_XGB_v3' — cv_oof_rmse: 401.24, cv_oof_mape: 3.48
🏃 View run Trend_XGB_v3 at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2/runs/6f7f163064d042eb8094436a8e80c9a3
🧪 View experiment at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2
--- lgbm_v3 ---
Fold 1 RMSE: 404.51
Fold 2 RMSE: 386.38
Fold 3 RMSE: 390.29
Fold 4 RMSE: 429.65
Fold 5 RMSE: 384.63
Trend+LightGBM (v3), CV OOF RMSE: 399.46  MAPE: 3.48%

Logged 'Trend_LightGBM_v3' — cv_oof_rmse: 399.46, cv_oof_mape: 3.48
🏃 View run Trend_LightGBM_v3 at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2/runs/d139f1faafc4431d94a1694f130f3179
🧪 View experiment at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2

## Model Comparison — RMSE / MAPE / Peak-MAPE

In [25]:
oof_dict_v3 = {name: v3_results[name]["oof"] for name in model_configs}
oof_dict_v3["lr"] = result_lr["oof"]  # note: lr ran on raw FEATURES, not v3 — included for reference only

model_rows = []
for name, oof in oof_dict_v3.items():
    valid = ~np.isnan(oof)
    y_true, y_pred = train.loc[valid, config.TARGET], oof[valid]
    m = cv.compute_metrics(y_true.to_numpy(), y_pred)
    model_rows.append({"model": name, **m})

model_comparison = pd.DataFrame(model_rows).sort_values("peak_mape").reset_index(drop=True)
model_comparison


,model,rmse,mape,peak_mape
0,lgbm_v3,399.462778,3.482224,4.023537
1,catboost_v3,394.272088,3.457084,4.031536
2,xgb_v3,401.242833,3.480206,4.076747
3,knn_v3,556.953316,5.162665,4.312551
4,lr,762.381695,6.468634,5.757238


## Residual Correlation Check

In [26]:
common_valid = np.ones(len(train), dtype=bool)
for oof in oof_dict_v3.values():
    common_valid &= ~np.isnan(oof)

resid_df = pd.DataFrame({
    name: (train[config.TARGET].values - oof)[common_valid]
    for name, oof in oof_dict_v3.items()
})
print(f"Rows compared: {common_valid.sum():,}")
resid_df.corr()


Rows compared: 43,824


,xgb_v3,lgbm_v3,catboost_v3,knn_v3,lr
xgb_v3,1.000000,0.963335,0.910203,0.636221,0.438522
lgbm_v3,0.963335,1.000000,0.923620,0.649987,0.446224
catboost_v3,0.910203,0.923620,1.000000,0.697187,0.486682
knn_v3,0.636221,0.649987,0.697187,1.000000,0.474338
lr,0.438522,0.446224,0.486682,0.474338,1.000000


## Hill-Climbing Ensemble

In [27]:
y_true_full = train[config.TARGET].values
ensemble_preds, selected, ensemble_weights, ensemble_history, solo_rmse = cv.hill_climb_ensemble(
    oof_dict_v3, y_true_full, common_valid
)

print("Solo RMSE (on common_valid rows):")
for name, rmse in sorted(solo_rmse.items(), key=lambda x: x[1]):
    print(f"  {name}: {rmse:.2f}")

print(f"\nHill-climb ensemble OOF RMSE: {ensemble_history[-1]:.2f}")
print("\nModel weights (by selection frequency):")
print(ensemble_weights)


Solo RMSE (on common_valid rows):
  catboost_v3: 394.27
  lgbm_v3: 399.46
  xgb_v3: 401.24
  knn_v3: 556.95
  lr: 762.38

Hill-climb ensemble OOF RMSE: 387.67

Model weights (by selection frequency):
catboost_v3    0.50
xgb_v3         0.25
lgbm_v3        0.25
Name: proportion, dtype: float64


## Hyperparameter Tuning — XGBoost / LightGBM / CatBoost (Optuna)
Each `run_study` call replaces a separate hand-written objective + CV loop — same shared `oof_rmse_for_model` harness underneath for all three.

In [ ]:
studies = {}
for model_name in ("xgb", "lgbm", "catboost"):
    baseline_rmse = v3_metrics[f"{model_name}_v3"]["rmse"]
    study = hpo.run_study(
        model_name, train, config.FOLD_BOUNDARIES, config.TARGET, config.FEATURES_V3,
        fold_feature_fn=v3_fold_features, n_trials=40,
    )
    studies[model_name] = study
    print(f"Best {model_name} OOF RMSE: {study.best_value:.2f}  (default-params baseline: {baseline_rmse:.2f})")
    print(study.best_params)


## Tuned Model Comparison

In [29]:
tuned_comparison = pd.DataFrame([
    {"model": name, "tuned_oof_rmse": study.best_value}
    for name, study in studies.items()
]).sort_values("tuned_oof_rmse").reset_index(drop=True)
tuned_comparison


,model,tuned_oof_rmse
0,catboost,389.480161
1,lgbm,390.847332
2,xgb,391.317409


## Target-Encoding Check — Holiday Identity

In [30]:
CHAMPION_NAME = tuned_comparison.iloc[0]["model"]
CHAMPION_OOF_RMSE = tuned_comparison.iloc[0]["tuned_oof_rmse"]
CHAMPION_PARAMS = studies[CHAMPION_NAME].best_params

champion_fold_features = features.compose(features.extreme_event_fold_features, features.holiday_freq_fold_features)
champion_model_fn = lambda: hpo.MODEL_BUILDERS[CHAMPION_NAME](CHAMPION_PARAMS)

holiday_oof_rmse = hpo.oof_rmse_for_model(
    train, config.FOLD_BOUNDARIES, config.TARGET, config.FEATURES_V3 + ["holiday_freq"],
    champion_model_fn, champion_fold_features,
)

print(f"Champion ({CHAMPION_NAME}) OOF RMSE without holiday_freq: {CHAMPION_OOF_RMSE:.2f}")
print(f"Champion ({CHAMPION_NAME}) OOF RMSE with holiday_freq:    {holiday_oof_rmse:.2f}")

USE_HOLIDAY_FEATURE = holiday_oof_rmse < CHAMPION_OOF_RMSE
print(f"\nUse holiday_freq in final model: {USE_HOLIDAY_FEATURE}")


Champion (catboost) OOF RMSE without holiday_freq: 389.48
Champion (catboost) OOF RMSE with holiday_freq:    389.73

Use holiday_freq in final model: False


## Champion Selection & Final Model Fit
Same fit/apply feature functions as every CV fold above — only fit on the full `train` set this time instead of a fold slice.

In [31]:
WINNING_FEATURES = config.FEATURES_V3 + ["holiday_freq"] if USE_HOLIDAY_FEATURE else config.FEATURES_V3

thresholds = features.fit_extreme_event_thresholds(train)
train_final = features.apply_extreme_event_features(train, thresholds)
test_final = features.apply_extreme_event_features(test, thresholds)

holiday_freq_map = None
if USE_HOLIDAY_FEATURE:
    holiday_freq_map = features.fit_holiday_freq_map(train_final)
    train_final = features.apply_holiday_freq(train_final, holiday_freq_map)
    test_final = features.apply_holiday_freq(test_final, holiday_freq_map)

X_train_final, y_train_final = train_final[WINNING_FEATURES], train_final[config.TARGET]
X_test_final, y_test_final = test_final[WINNING_FEATURES], test_final[config.TARGET]

trend_model_final = LinearRegression().fit(train_final[["trend_idx"]], y_train_final)
trend_train_final = trend_model_final.predict(train_final[["trend_idx"]])
trend_test_final = trend_model_final.predict(test_final[["trend_idx"]])

residual_target_final = y_train_final - trend_train_final

final_model = hpo.MODEL_BUILDERS[CHAMPION_NAME](CHAMPION_PARAMS)
final_model.fit(X_train_final, residual_target_final)
pred_test_final = trend_test_final + final_model.predict(X_test_final)

test_metrics = cv.compute_metrics(y_test_final.to_numpy(), pred_test_final)

print(f"Champion model: {CHAMPION_NAME}  |  Features: {'v3 + holiday_freq' if USE_HOLIDAY_FEATURE else 'v3'}")
print(f"Final test RMSE: {test_metrics['rmse']:.2f}")
print(f"Final test MAPE: {test_metrics['mape']:.3f}%")
print(f"Final test Peak-MAPE: {test_metrics['peak_mape']:.3f}%")


Champion model: catboost  |  Features: v3
Final test RMSE: 360.73
Final test MAPE: 2.898%
Final test Peak-MAPE: 3.512%


## MLflow Logging — Champion Model

In [32]:
import joblib
from pathlib import Path
import mlflow
import mlflow.pyfunc


class ElectricityForecaster(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        self.bundle = joblib.load(context.artifacts["model_bundle"])
        self.trend_model = self.bundle["trend_model"]
        self.residual_model = self.bundle["residual_model"]
        self.features = self.bundle["features"]

    def predict(self, context, model_input):
        df = model_input if isinstance(model_input, pd.DataFrame) else pd.DataFrame(model_input)
        trend_preds = self.trend_model.predict(df[["trend_idx"]])
        residual_preds = self.residual_model.predict(df[self.features])
        return trend_preds + residual_preds


artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)
model_path = artifact_dir / "electricity_load_forecaster.pkl"

production_bundle = {
    "trend_model": trend_model_final,
    "residual_model": final_model,
    "features": WINNING_FEATURES,
    "target": config.TARGET,
    "thresholds": thresholds,
    "holiday_freq_map": holiday_freq_map,  # None if USE_HOLIDAY_FEATURE is False
    "model_family": f"trend_plus_tuned_{CHAMPION_NAME}",
    "feature_version": "v3_holiday" if USE_HOLIDAY_FEATURE else "v3",
    "train_end": str(train_final["timestamp"].max()),
    "test_start": str(test_final["timestamp"].min()),
}
joblib.dump(production_bundle, model_path)

with mlflow.start_run(run_name="production_candidate_final"):
    mlflow.log_params({
        "model_family": production_bundle["model_family"],
        "feature_version": production_bundle["feature_version"],
        **{f"{CHAMPION_NAME}_{k}": v for k, v in CHAMPION_PARAMS.items()},
        "train_rows": len(train_final),
        "test_rows": len(test_final),
        "purge_days": config.PURGE_DAYS,
        "train_end": production_bundle["train_end"],
        "test_start": production_bundle["test_start"],
    })
    mlflow.log_metrics({
        "cv_oof_rmse": float(CHAMPION_OOF_RMSE),
        "test_rmse": float(test_metrics["rmse"]),
        "test_mape_pct": float(test_metrics["mape"]),
        "test_peak_mape_pct": float(test_metrics["peak_mape"]),
    })

    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=ElectricityForecaster(),
        artifacts={"model_bundle": str(model_path)},
        registered_model_name="Electricity-Load-Forecaster",
    )

print("Successfully logged params, metrics, and registered the production candidate model!")


/teamspace/studios/this_studio/electricity-distribution-forecast/.venv/lib/python3.14/site-packages/mlflow/types/type_hints.py:232: UserWarning: Any type hint is inferred as AnyType, and MLflow doesn't validate the data for this type. Please use a more specific type hint to enable data validation.
  dtype=_infer_colspec_type_from_type_hint(effective_type).dtype,
/teamspace/studios/this_studio/electricity-distribution-forecast/.venv/lib/python3.14/site-packages/mlflow/types/type_hints.py:213: UserWarning: Any type hint is inferred as AnyType, and MLflow doesn't validate the data for this type. Please use a more specific type hint to enable data validation.
  dtype=Map(_infer_colspec_type_from_type_hint(type_hint=args[1]).dtype),
/teamspace/studios/this_studio/electricity-distribution-forecast/.venv/lib/python3.14/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during

2026/08/14 22:29:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/14 22:29:12 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
2026/08/14 22:29:16 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'Electricity-Load-Forecaster' already exists. Creating a new version of this model...
2026/08/14 22:29:18 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Electricity-Load-Forecaster, version 4
Created version '4' of model 'Electricity-Load-For

🏃 View run production_candidate_final at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2/runs/8fe77a5b95274e76b1766ad4819e677c
🧪 View experiment at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2
Successfully logged params, metrics, and registered the production candidate model!


In [33]:
sample = test[fe.RAW_REQUIRED_COLUMNS].sample(20, random_state=42)
sample.to_parquet("../data/interim/pipeline_test_sample.parquet", index=False)

In [2]:
import mlflow
print(mlflow.pyfunc.get_model_dependencies("models:/Electricity-Load-Forecaster@champion"))

/teamspace/studios/this_studio/electricity-distribution-forecast/.venv/lib/python3.14/site-packages/mlflow/types/type_hints.py:232: UserWarning: Any type hint is inferred as AnyType, and MLflow doesn't validate the data for this type. Please use a more specific type hint to enable data validation.
  dtype=_infer_colspec_type_from_type_hint(effective_type).dtype,
/teamspace/studios/this_studio/electricity-distribution-forecast/.venv/lib/python3.14/site-packages/mlflow/types/type_hints.py:213: UserWarning: Any type hint is inferred as AnyType, and MLflow doesn't validate the data for this type. Please use a more specific type hint to enable data validation.
  dtype=Map(_infer_colspec_type_from_type_hint(type_hint=args[1]).dtype),
2026/08/15 18:02:19 INFO mlflow.pyfunc: To install the dependencies that were used to train the model, run the following command: '%pip install -r /tmp/tmpi1x1ur18/requirements.txt'.


/tmp/tmpi1x1ur18/requirements.txt
